In [3]:
import subprocess

print("=" * 60)
print("GPU CHECK (nvidia-smi)")
print("=" * 60)
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

GPU CHECK (nvidia-smi)
Mon Aug 31 10:04:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------

In [2]:
import torch

print("=" * 60)
print("PYTORCH / CUDA CHECK")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version (PyTorch built with): {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU compute capability: {torch.cuda.get_device_capability(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total VRAM: {total_vram:.2f} GB")
else:
    print("⚠️ No CUDA GPU detected — check NVIDIA drivers and PyTorch install")

PYTORCH / CUDA CHECK
PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA version (PyTorch built with): 12.8
GPU name: Tesla T4
GPU compute capability: (7, 5)
Total VRAM: 15.64 GB


In [4]:
!pip install einops ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 9.7 MB/s eta 0:00:00


In [5]:
import subprocess

result = subprocess.run(
    ['pip', 'install', 'flash-attn-3', '--no-deps',
     '--index-url', 'https://download.pytorch.org/whl/cu128'],
    capture_output=True, text=True
)

print("STDOUT:")
print(result.stdout[-3000:])  # last part of output, most relevant
print("\nSTDERR:")
print(result.stderr[-3000:])
print(f"\nReturn code: {result.returncode}")

STDOUT:
Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.2/442.2 MB 3.9 MB/s eta 0:00:00


STDERR:


Return code: 0


In [6]:
try:
    import flash_attn_3
    print("✓ flash_attn_3 imported successfully")
    print("Version:", getattr(flash_attn_3, '__version__', 'unknown'))
except ImportError as e:
    print("✗ flash_attn_3 import FAILED")
    print(f"Error: {e}")
except Exception as e:
    print("✗ flash_attn_3 imported but errored")
    print(f"Error: {e}")

✓ flash_attn_3 imported successfully
Version: unknown


In [7]:
print("=" * 60)
print("VRAM BUDGET ESTIMATE (rough)")
print("=" * 60)

if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    sam3_weights_fp16_gb = 1.7  # ~840M params in fp16
    estimated_overhead_gb = 1.5  # CUDA context, PyTorch overhead, etc.
    
    remaining = total_vram_gb - sam3_weights_fp16_gb - estimated_overhead_gb
    
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
    print(f"SAM-3 weights (fp16, frozen encoder still loaded): ~{sam3_weights_fp16_gb} GB")
    print(f"Estimated PyTorch/CUDA overhead: ~{estimated_overhead_gb} GB")
    print(f"Remaining for decoder training (activations, gradients, batch): ~{remaining:.2f} GB")
    
    if remaining < 1.0:
        print("\n⚠️ WARNING: Very tight budget. May need batch_size=1, gradient checkpointing, low resolution.")
    elif remaining < 2.5:
        print("\n✓ Workable but tight — decoder-only fine-tuning with small batch size should fit.")
    else:
        print("\n✓ Reasonable headroom for decoder-only fine-tuning.")
else:
    print("No GPU detected — cannot estimate.")

VRAM BUDGET ESTIMATE (rough)
Total VRAM: 15.64 GB
SAM-3 weights (fp16, frozen encoder still loaded): ~1.7 GB
Estimated PyTorch/CUDA overhead: ~1.5 GB
Remaining for decoder training (activations, gradients, batch): ~12.44 GB

✓ Reasonable headroom for decoder-only fine-tuning.


In [8]:
import torch
print("Device name:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

Device name: Tesla T4
Compute capability: (7, 5)
